In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio langchain langchain-huggingface

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-7B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
import asyncio, json, re, logging, time
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from transformers import pipeline

# 1. Configurar pipeline básico de generación de texto
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# 2. Envolver con LangChain pasando max_new_tokens a través de pipeline_kwargs
hf_llm = HuggingFacePipeline(pipeline=pipe, pipeline_kwargs={"max_new_tokens": 1024})
chat_model = ChatHuggingFace(llm=hf_llm)

class LangChainAgent:
    def __init__(self, model):
        self.model = model

    def complete(self, messages):
        lc_messages = []
        for msg in messages:
            if msg.role == 'system':
                lc_messages.append(SystemMessage(content=msg.content))
            elif msg.role == 'user':
                lc_messages.append(HumanMessage(content=msg.content))
            elif msg.role == 'assistant':
                lc_messages.append(AIMessage(content=msg.content))
        
        response_msg = self.model.invoke(lc_messages)
        response = response_msg.content
        return re.sub(r'<\|im_end\|>', '', response)


In [ ]:
llm = LangChainAgent(chat_model)

from pydantic import BaseModel
from typing import List, Optional, Literal
from datetime import datetime

class Message(BaseModel):
    role: Literal['system', 'user']  # roles comunes en chats
    content: str

class ChatRequest(BaseModel):
    messages: List[Message] = []  # Valor por defecto lista vacía



In [ ]:
from fastapi import FastAPI, Request
from pyngrok import ngrok
import nest_asyncio
import uvicorn

ngrok.set_auth_token("344HT0PzWr1pGVLwZBa7KWXfxXE_4FMsfMKfHFpG8ZAQXrpS7")


nest_asyncio.apply()  # Permite correr uvicorn en el loop de Colab

app = FastAPI()


@app.get("/")
def home():
    return {"message": "Hola desde Colab + FastAPI!"}



@app.post("/chat")
async def chat(body: ChatRequest):

    response = llm.complete(body.messages)
    print(response)

    return {
        "response": response
    }

In [ ]:
from pyngrok import ngrok

# Cerrar cualquier túnel previo colgado para evitar el error ERR_NGROK_334
try:
    print("Cerrando túneles activos anteriores...")
    ngrok.kill()
except Exception as e:
    print("No se encontraron túneles previos activos.")

# Crear túnel en el puerto 8000
public_url = ngrok.connect(8000)
print("URL pública:", public_url)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()
